# Chapter 3 — Claude Agent SDK: Building Agentic Systems · Homework (self-test)

**Goal:** check that you can design, configure, and guard a multi-agent system with the Claude Agent SDK, the way the exam's Domain 1 scenarios expect.

- Theory: [README.md](README.md) · Practice: [01_practice.ipynb](01_practice.ipynb)
- Estimated time: 90–120 minutes
- Estimated API cost: **$0**. Everything is graded offline. The optional live stretch at the end of Part B costs about $0.05–$0.15.
- Prerequisites: finish the practice notebook, and run this notebook from the repo venv (see [00-prerequisites](../../00-prerequisites/README.md))

**How to use it**

1. Attempt everything **without peeking**. Solutions are at the very bottom, in collapsed blocks.
2. **Part A:** 12 scenario questions. Put your letters in the `answers` dict and run the grader.
3. **Part B:** 7 coding exercises. Replace each `TODO`, then run its check cell. Checks print `✓` or a hint and never crash the notebook.
4. **Part C:** one architecture scenario, written answer, self-assessed against a rubric.
5. **Scorecard:** the overall score is the average of quiz % and exercises %. The pass mark is **72%**, which mirrors the Claude exam's 720/1000.

In [ ]:
# Setup (offline)
import asyncio
import copy
import dataclasses
import hashlib
import json
import os
from types import SimpleNamespace
from typing import Any

from anthropic.types import Message, Usage
from anthropic.types import TextBlock as ApiTextBlock, ToolUseBlock as ApiToolUseBlock
from claude_agent_sdk import (
    AgentDefinition, AssistantMessage, ClaudeAgentOptions, HookMatcher, PermissionResultAllow, PermissionResultDeny,
    ResultMessage, SystemMessage, ToolPermissionContext, ToolResultBlock, ToolUseBlock, UserMessage,
    create_sdk_mcp_server, query, tool,
)
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv())
MODEL = os.getenv("CLAUDE_MODEL", "claude-sonnet-5-5")
RESULTS: dict[str, Any] = {}   # filled by the graders; read by the scorecard


def require_live() -> None:
    if not os.getenv("ANTHROPIC_API_KEY"):
        raise RuntimeError("Live cell skipped: set ANTHROPIC_API_KEY in .env (see 00-prerequisites).")


def _record(ex: str, fn) -> bool:
    """Run one checker; never let it crash the notebook."""
    try:
        fn()
    except NotImplementedError:
        print(f"✗ {ex}: not implemented yet")
        RESULTS[ex] = False
    except AssertionError as err:
        print(f"✗ {ex}: {err}")
        RESULTS[ex] = False
    except Exception as err:  # noqa: BLE001
        print(f"✗ {ex}: {type(err).__name__}: {err}")
        RESULTS[ex] = False
    else:
        print(f"✓ {ex.replace('ex', 'Exercise ')} passed")
        RESULTS[ex] = True
    return RESULTS[ex]


async def _record_async(ex: str, coro_fn) -> bool:
    try:
        await coro_fn()
    except NotImplementedError:
        print(f"✗ {ex}: not implemented yet")
        RESULTS[ex] = False
    except AssertionError as err:
        print(f"✗ {ex}: {err}")
        RESULTS[ex] = False
    except Exception as err:  # noqa: BLE001
        print(f"✗ {ex}: {type(err).__name__}: {err}")
        RESULTS[ex] = False
    else:
        print(f"✓ {ex.replace('ex', 'Exercise ')} passed")
        RESULTS[ex] = True
    return RESULTS[ex]


print("model:", MODEL)

In [ ]:
# Key check: only needed for the optional live stretch at the end of Part B
assert os.getenv("ANTHROPIC_API_KEY"), "Set ANTHROPIC_API_KEY in .env (see 00-prerequisites)"
print("API key found.")

## Part A — Quiz

Each question has exactly one best answer. The scenarios mirror the exam style: pick the option that fixes the **root cause** with the right mechanism.

**Q1.** Your Agent SDK app iterates `query()`. A teammate's code `break`s out of the `async for` loop as soon as an `AssistantMessage` contains the word "complete", and reports success. Some runs are reported as successful even though the run actually ended with `error_max_turns`. What is the best fix?

- A) Raise `max_turns` so that runs rarely hit the limit
- B) Add "Only say 'complete' when you are truly finished" to the system prompt
- C) Iterate the stream to completion and decide success from `ResultMessage.subtype == "success"`, treating every other subtype as a failure
- D) Treat a run as successful when its last `AssistantMessage` contains a text block

**Q2.** A headless review agent is configured with `allowed_tools=["Read", "Grep"]` and `permission_mode="bypassPermissions"`. During a run it calls `Bash` and deletes a temp folder. Why?

- A) `allowed_tools` only pre-approves the listed tools. It doesn't restrict the others, and `bypassPermissions` approves every unlisted call that reaches the permission-mode check
- B) `Bash` is always exempt from `allowed_tools`
- C) The `Agent` tool re-enabled `Bash` for the main agent
- D) `allowed_tools` is ignored unless `"Agent"` is also listed

**Q3.** To "keep it simple", you define `"report-writer": AgentDefinition(description=..., prompt=...)` with no `tools` field. In testing, the report writer starts running shell commands and editing files. What explains this, and what is the fix?

- A) Subagents always receive `Bash`; tell the writer in its prompt not to use it
- B) The coordinator's `allowed_tools` leaks into subagents; remove `Bash` from it
- C) `model` was omitted, so the writer inherited a more capable model; set `model="haiku"`
- D) Omitting `tools` makes the subagent inherit every tool available to subagents; list only what it needs, e.g. `tools=["Read"]`

**Q4.** Which configuration registers a subagent correctly with the current Python Claude Agent SDK?

- A) `ClaudeAgentOptions(agents=[AgentDefinition(name="reviewer", system_prompt="...", allowed_tools=["Read"])])`
- B) `ClaudeAgentOptions(agents={"reviewer": AgentDefinition(description="...", prompt="...", tools=["Read"], maxTurns=5)})`
- C) `ClaudeAgentOptions(agents={"reviewer": AgentDefinition(description="...", prompt="...", tools=["Read"], max_turns=5)})`
- D) `ClaudeAgentOptions(subagents={"reviewer": {"description": "...", "system_prompt": "..."}})`

**Q5.** Over 10 turns, a coordinator established that the customer is on an Enterprise plan with a custom 2-hour SLA. It then delegated "Draft the incident response email" to a `writer` subagent. The draft quotes the standard 24-hour SLA. What is the most effective fix?

- A) Switch the writer to a larger model
- B) Add "remember what the coordinator learned earlier" to the writer's `AgentDefinition.prompt`
- C) Put the plan, the custom SLA terms and the incident facts (structured, verbatim) in the prompt of the `Agent` tool call
- D) Raise the writer's `maxTurns` so it has time to find the SLA itself

**Q6.** You must block `mcp__billing__refund` only when `amount > 500`. A teammate registers `HookMatcher(matcher="mcp__billing__refund(amount>500)", hooks=[deny_all])`. What is the correct approach?

- A) Matchers only test the tool name. Use `matcher="mcp__billing__refund"` and check `input_data["tool_input"]["amount"]` inside the callback
- B) The syntax is right, but the matcher needs a `^` anchor
- C) Use a `PostToolUse` hook, because the amount is only known after the call
- D) Put the threshold in the `timeout` field of `HookMatcher`

**Q7.** Two `PreToolUse` hooks match `mcp__support__process_refund`. An auto-approver returns `permissionDecision: "allow"`, and a limit check returns `"deny"`. The session runs in `bypassPermissions` mode. What happens?

- A) Whichever hook finishes first wins
- B) The call runs, because `bypassPermissions` overrides hook decisions
- C) Claude is asked to choose between the two decisions
- D) The call is blocked: `deny` takes precedence over every other decision, and a hook deny applies even in `bypassPermissions`

**Q8.** In a research system, three researcher subagents keep fetching the same top sources. Tokens are wasted, but the final report is fine. What is the best fix?

- A) Add a deduplication step after research finishes
- B) Have the coordinator partition the scope (distinct subtopics or source types) **before** delegating, and state each partition in each subagent's prompt
- C) Let the researchers message each other directly to coordinate
- D) Merge them into a single researcher that holds every tool

**Q9.** A search subagent finds **0 documents** for "AI dubbing regulation in Brazil". Separately, its call to the news API **times out**. What should it return to the coordinator?

- A) `{"status": "success", "results": []}` for both
- B) A single generic error: "search failed"
- C) The empty search as a valid finding (status ok, 0 results, the query), and the timeout as a structured failure (failure type, attempted query, partial results, alternatives)
- D) Raise an exception so that the coordinator restarts the whole workflow

**Q10.** Yesterday an agent analyzed a repository over 40 turns. Overnight, a large refactor changed most of the files it had read. Today you want to continue planning the migration. What is the most reliable approach?

- A) `resume=session_id` and continue as if nothing changed
- B) Start a new session seeded with a structured summary of yesterday's conclusions plus a list of what changed, so that the agent re-reads current files
- C) Resume with `fork_session=True` so that the original session stays intact
- D) Use `continue_conversation=True` with a higher `max_turns`

**Q11.** A nightly headless agent runs with no `permission_mode` set, and no settings file sets `permissions.defaultMode`. In testing it sometimes runs shell commands that nobody pre-approved, while a broad `Bash` allow rule seems to be ignored. What is the most likely explanation, and what is the fix?

- A) `allowed_tools` is ignored in headless runs; move every rule into hooks
- B) `Bash` is always exempt from permission rules; remove it with `disallowed_tools`
- C) Headless runs always use `bypassPermissions`; switch to `ClaudeSDKClient`
- D) With the mode unset, the session can start in `auto` mode, where a classifier approves or blocks actions and broad allow rules such as a bare `Bash` are dropped; set `permission_mode` explicitly, for example `"dontAsk"` with an explicit `allowed_tools` list

**Q12.** To "speed things up", a `PreToolUse` hook returns `permissionDecision: "allow"` for every `Bash` call. The options also contain `disallowed_tools=["Bash(rm *)"]`. Claude tries `rm -rf build`. What happens?

- A) The call is blocked: a hook `allow` doesn't skip deny rules, and the scoped deny rule still matches
- B) The call runs: hooks run first, so their `allow` ends the evaluation
- C) The call runs only if `permission_mode="bypassPermissions"`
- D) `can_use_tool` is called to break the tie between the hook and the rule

In [ ]:
# Your answers: use "A", "B", "C" or "D"
answers = {
    "q1": None, "q2": None, "q3": None, "q4": None, "q5": None,
    "q6": None, "q7": None, "q8": None, "q9": None, "q10": None,
    "q11": None, "q12": None,
}

In [ ]:
# Quiz grader (answers are stored as hashes, so reading this cell doesn't spoil anything)
_QUIZ_KEY = {
    "q1": "227effd0f0211c628c0ae190f31feffeed3e6aa4b570e9bd60b91ac5bb2bb16e",
    "q2": "3d9c97ee45947882f5eee5cf878c4f33a0a270d8f2a2e1fc259d66cdc95c7b45",
    "q3": "14099ffd7333e442b90669f0e435e505e0e45a0ed148540caca94158c15a27c6",
    "q4": "61f643a360ded8f53a0309d76dab6c7178f5a30db61b94cdcd57e7341a5aeaaa",
    "q5": "e3c6d5ddd445920945724cd165c914deaf97d5041f947ba2df35e569269b353f",
    "q6": "ac78c6455837b3b074080e7912c1bdab72fbbfdcaeefd9269fe293a50c4a9522",
    "q7": "45ff49923feb785985244958470606d87fb75c9912783e470a079e3bb46e1457",
    "q8": "39f95f59afaa9522616af26bd34df1e0184030ae8f738e319fc9a1f15b075834",
    "q9": "e19608e84d37f3c94b5f88738c75a8c96772a5e61784e3e24b5512fa65ff63c7",
    "q10": "c69a9867de9de4cc2e45f4cd6a6128833af714d8993edbeac622795c9c5e20d3",
    "q11": "19b8f64da90dd1e288d2fa60f4128e5e8c665ace09e8470b263486752199e135",
    "q12": "9d4e72c9d2036d777ce76bca900a85c706c8a653c712a6faf259d63490f850a2",
}


def grade_quiz(answers: dict) -> float:
    correct = 0
    for qid, expected in _QUIZ_KEY.items():
        given = answers.get(qid)
        ok = isinstance(given, str) and hashlib.sha256(f"{qid}:{given.strip().upper()}".encode()).hexdigest() == expected
        correct += ok
        print(f"{qid:>4}: {'✓' if ok else '✗'}{'' if given else '  (unanswered)'}")
    pct = 100 * correct / len(_QUIZ_KEY)
    RESULTS["quiz_pct"] = pct
    print(f"\nQuiz: {correct}/{len(_QUIZ_KEY)} = {pct:.0f}%")
    return pct


grade_quiz(answers)

## Part B — Coding exercises

All seven are graded offline: you build configs, write hooks, and implement control flow against fake objects. The SDK's own classes (`AgentDefinition`, `ClaudeAgentOptions`, message types) are real, so a wrong field name fails the same way it would in production.

### Exercise 1 — Define two subagents with `AgentDefinition`

Write `build_research_agents()` returning a dict with **exactly** two keys:

| Key | `tools` | `model` | Other requirements |
|---|---|---|---|
| `"web-researcher"` | exactly `WebSearch`, `WebFetch` | `"haiku"` | `maxTurns` between 1 and 10 |
| `"synthesizer"` | exactly `["Read"]` | `"inherit"` | — |

For both: a `description` of at least 10 words that says **when** to use the agent (the two must not overlap much), a `prompt` that states a **JSON** output format, and no ability to spawn further subagents.

In [ ]:
# Exercise 1: your code
def build_research_agents() -> dict[str, AgentDefinition]:
    # TODO: return {"web-researcher": AgentDefinition(...), "synthesizer": AgentDefinition(...)}
    raise NotImplementedError

In [ ]:
# Exercise 1: check
def _check_ex1():
    agents = build_research_agents()
    assert isinstance(agents, dict) and set(agents) == {"web-researcher", "synthesizer"}, \
        "return a dict with exactly the keys 'web-researcher' and 'synthesizer' (the key IS the name)"
    for name, a in agents.items():
        assert isinstance(a, AgentDefinition), f"{name}: use AgentDefinition(...)"
        assert a.tools is not None, f"{name}: set `tools`, because omitting it inherits every tool"
        assert not {"Agent", "Task"} & set(a.tools), f"{name}: subagents must not spawn subagents here"
        assert len(a.description.split()) >= 10, f"{name}: description needs >= 10 words to route well"
        assert a.prompt and "json" in a.prompt.lower(), f"{name}: the prompt must state a JSON output format"
    r, s = agents["web-researcher"], agents["synthesizer"]
    assert set(r.tools) == {"WebSearch", "WebFetch"}, "web-researcher tools must be exactly WebSearch + WebFetch"
    assert r.model == "haiku", "web-researcher should use the 'haiku' alias"
    assert isinstance(r.maxTurns, int) and 1 <= r.maxTurns <= 10, "web-researcher needs maxTurns between 1 and 10 (camelCase!)"
    assert s.tools == ["Read"], "synthesizer tools must be exactly ['Read']"
    assert s.model == "inherit", "synthesizer should use model='inherit'"
    wr = {w for w in r.description.lower().split() if len(w) > 3}
    ws = {w for w in s.description.lower().split() if len(w) > 3}
    assert len(wr & ws) / max(1, len(wr | ws)) < 0.5, "descriptions overlap too much, so the coordinator will misroute"


check_ex1 = lambda: _record("ex1", _check_ex1)  # noqa: E731
check_ex1()

### Exercise 2 — Configure the coordinator with `ClaudeAgentOptions`

Write `build_coordinator_options(agents)` that returns `ClaudeAgentOptions` with:

- `model=MODEL`, and a goal-oriented `system_prompt` (at least 10 words)
- the `agents` dict passed through unchanged
- `"Agent"` pre-approved in `allowed_tools`, and `"Bash"` **not** pre-approved
- `"Bash"` removed entirely with a bare-name `disallowed_tools` entry
- `permission_mode="dontAsk"`, so unlisted calls are denied instead of prompted
- a safety cap `max_turns` (1–30) and a cost cap `0 < max_budget_usd <= 2.0`
- nested spawning disabled with the environment variable `CLAUDE_CODE_MAX_SUBAGENT_SPAWN_DEPTH` set to `"1"` (through the `env` option)

In [ ]:
# Exercise 2: your code
def build_coordinator_options(agents: dict[str, AgentDefinition]) -> ClaudeAgentOptions:
    # TODO: return ClaudeAgentOptions(...) meeting every requirement above
    raise NotImplementedError

In [ ]:
# Exercise 2: check
def _check_ex2():
    agents = {"x": AgentDefinition(description="d", prompt="p", tools=["Read"])}
    o = build_coordinator_options(agents)
    assert isinstance(o, ClaudeAgentOptions), "return ClaudeAgentOptions(...)"
    assert o.model == MODEL, "use model=MODEL"
    assert isinstance(o.system_prompt, str) and len(o.system_prompt.split()) >= 10, \
        "give the coordinator a real system prompt (goals + quality criteria, >= 10 words)"
    assert o.agents is agents, "pass the agents dict through unchanged"
    assert "Agent" in o.allowed_tools, "pre-approve the subagent tool: 'Agent'"
    assert "Bash" not in o.allowed_tools, "never auto-approve Bash for this coordinator"
    assert "Bash" in o.disallowed_tools, "remove Bash with a bare-name disallowed_tools entry"
    assert o.permission_mode == "dontAsk", "use permission_mode='dontAsk' so unlisted calls are denied, not prompted"
    assert o.max_budget_usd is not None and 0 < o.max_budget_usd <= 2.0, "set 0 < max_budget_usd <= 2.0"
    assert isinstance(o.max_turns, int) and 1 <= o.max_turns <= 30, "set max_turns (1..30) as a safety cap"
    assert o.env.get("CLAUDE_CODE_MAX_SUBAGENT_SPAWN_DEPTH") == "1", \
        "cap nesting with env={'CLAUDE_CODE_MAX_SUBAGENT_SPAWN_DEPTH': '1'} (a string)"


check_ex2 = lambda: _record("ex2", _check_ex2)  # noqa: E731
check_ex2()

### Exercise 3 — Least-privilege tool assignment

A support system has four roles. From `TOOL_MENU`, give each role the **smallest** set that lets it do its job:

- **coordinator:** only delegates to specialists and talks to the customer. It may escalate a case to a human itself.
- **order-specialist:** verifies the customer's identity and looks up orders. It never moves money.
- **refund-specialist:** re-checks the order total, issues refunds, and must be able to escalate when policy blocks a refund.
- **reply-writer:** drafts the customer reply from findings passed in its prompt. It may spot-check a single policy fact but must not browse the web.

Return `dict[str, set[str]]` keyed by role. The checker stores the expected sets as hashes, so it can only tell you which role is off.

In [ ]:
# Exercise 3: your code
TOOL_MENU = {
    "Agent", "WebSearch", "WebFetch", "Read", "Bash",
    "mcp__crm__get_customer", "mcp__crm__lookup_order", "mcp__crm__process_refund",
    "mcp__crm__escalate_to_human", "mcp__kb__verify_fact",
}


def assign_tools() -> dict[str, set[str]]:
    # TODO: return {"coordinator": {...}, "order-specialist": {...}, "refund-specialist": {...}, "reply-writer": {...}}
    raise NotImplementedError

In [ ]:
# Exercise 3: check (the expected sets are hashed so they don't spoil the exercise)
_EX3_KEY = {
    "coordinator": "3c0b003495d1589e53a2592850b189ca9b11103f540695016818c42d590ba022",
    "order-specialist": "fa9ef7d6206788d7fd50226ffd773c7abab1d43de541350314c68c01f89a5810",
    "refund-specialist": "97dee068e5cb88c27788afd234c934934da7d894f214476788d718257479e29d",
    "reply-writer": "a97bdbacb6900daae59a2ac59161afcab4e5714a84a3aca8b6a3744ed7ddc3ba",
}
_EX3_HINTS = {
    "coordinator": "it only delegates and may escalate; it should not do the specialists' work itself",
    "order-specialist": "it verifies identity and reads orders; it never moves money",
    "refund-specialist": "it re-checks the total, refunds, and must be able to escalate when policy blocks it",
    "reply-writer": "it writes from findings in its prompt; one narrow fact-check tool, no web access",
}


def _h(tools) -> str:
    return hashlib.sha256(",".join(sorted(tools)).encode()).hexdigest()


def _check_ex3():
    plan = assign_tools()
    assert isinstance(plan, dict) and set(plan) == set(_EX3_KEY), f"return exactly these roles: {sorted(_EX3_KEY)}"
    for role, tools in plan.items():
        tools = set(tools)
        assert tools <= TOOL_MENU, f"{role}: unknown tool(s) {sorted(tools - TOOL_MENU)}"
        assert "Bash" not in tools, f"{role}: nobody in this system needs Bash"
        assert _h(tools) == _EX3_KEY[role], f"{role}: not least-privilege yet. Hint: {_EX3_HINTS[role]}"


check_ex3 = lambda: _record("ex3", _check_ex3)  # noqa: E731
check_ex3()

### Exercise 4 — A `PreToolUse` policy hook

Write the async callback `enforce_refund_policy(input_data, tool_use_id, context)` for `mcp__support__process_refund`:

- `amount` ≤ `REFUND_LIMIT_USD` (500): return `{}`
- `amount` > 500: **deny**, with a `permissionDecisionReason` that tells Claude to call `escalate_to_human`
- `amount` missing or not a number (it may arrive as a string such as `"750.00"`): **deny** (fail closed)
- put the decision inside `hookSpecificOutput` with `hookEventName: "PreToolUse"`, and never mutate `input_data`

You would register it as `HookMatcher(matcher="mcp__support__process_refund", hooks=[enforce_refund_policy])`.

In [ ]:
# Exercise 4: your code
REFUND_LIMIT_USD = 500


async def enforce_refund_policy(input_data: dict, tool_use_id: str | None, context: Any) -> dict:
    # TODO: deny > $500 (redirect to escalate_to_human), deny missing/invalid amounts, else return {}
    raise NotImplementedError

In [ ]:
# Exercise 4: check (async, so it is awaited)
def _fake_pre(tool_input: dict) -> dict:
    return {"hook_event_name": "PreToolUse", "session_id": "s1", "transcript_path": "", "cwd": "/tmp",
            "tool_name": "mcp__support__process_refund", "tool_input": tool_input, "tool_use_id": "toolu_1"}


async def _check_ex4():
    cases = [({"order_id": "A-1", "amount": 120.0}, "allow"), ({"order_id": "A-1", "amount": 500}, "allow"),
             ({"order_id": "A-1", "amount": 500.01}, "deny"), ({"order_id": "A-1", "amount": "750.00"}, "deny"),
             ({"order_id": "A-1"}, "deny"), ({"order_id": "A-1", "amount": "lots"}, "deny")]
    for tool_input, expected in cases:
        payload = _fake_pre(tool_input)
        before = copy.deepcopy(payload)
        out = await enforce_refund_policy(payload, "toolu_1", {"signal": None})
        assert payload == before, "don't mutate input_data / tool_input inside a hook"
        assert isinstance(out, dict), "a hook must return a dict ({} = let it through)"
        if expected == "allow":
            assert out == {}, f"{tool_input}: within policy, so return {{}}"
            continue
        assert "decision" not in out and "updatedInput" not in out, "put the decision inside hookSpecificOutput"
        hso = out.get("hookSpecificOutput", {})
        assert hso.get("hookEventName") == "PreToolUse", "set hookSpecificOutput.hookEventName = 'PreToolUse'"
        assert hso.get("permissionDecision") == "deny", f"{tool_input}: expected permissionDecision 'deny'"
        reason = hso.get("permissionDecisionReason", "")
        assert reason, "give Claude a permissionDecisionReason so it can adapt"
        if "amount" in tool_input and tool_input["amount"] != "lots":
            assert "escalate_to_human" in reason, "over-limit reason should redirect Claude to escalate_to_human"


async def check_ex4():
    return await _record_async("ex4", _check_ex4)


await check_ex4()

### Exercise 5 — The agentic loop, against a fake client

Implement `run_loop(llm, task, tools, impls, safety_cap=8)` with the Client SDK message format:

1. Start from `[{"role": "user", "content": task}]` and call `llm.messages.create(model=MODEL, max_tokens=512, tools=tools, messages=messages)`.
2. Always append the assistant turn (`resp.content`) to `messages`.
3. `stop_reason == "end_turn"`: return the joined text of the text blocks.
4. `stop_reason == "tool_use"`: run **every** `tool_use` block with `impls[name](**input)`, and append **one** user message with one `tool_result` per call (`tool_use_id`, `content=json.dumps(result)`).
5. Any other `stop_reason`: raise `RuntimeError`. Reaching `safety_cap` model calls: raise `RuntimeError`.

In [ ]:
# Helpers for Exercise 5: a scripted fake client (offline)
_n = iter(range(1, 10_000))


def fake_msg(stop_reason: str, *blocks) -> Message:
    return Message(id=f"msg_{next(_n)}", type="message", role="assistant", model="fake", content=list(blocks),
                   stop_reason=stop_reason, stop_sequence=None, usage=Usage(input_tokens=0, output_tokens=0))


def say(text: str) -> ApiTextBlock:
    return ApiTextBlock(type="text", text=text)


def call(name: str, tool_id: str, **tool_input) -> ApiToolUseBlock:
    return ApiToolUseBlock(type="tool_use", id=tool_id, name=name, input=tool_input)


class FakeLLM:
    """Replays scripted Messages and records the `messages` it was sent on every call."""

    def __init__(self, script):
        self._script, self.seen = list(script), []
        self.messages = SimpleNamespace(create=self._create)

    def _create(self, **kwargs) -> Message:
        self.seen.append(copy.deepcopy(kwargs["messages"]))
        if not self._script:
            raise AssertionError("the loop called the model again after the script ended")
        return self._script.pop(0)

In [ ]:
# Exercise 5: your code
def run_loop(llm, task: str, tools: list[dict], impls: dict, safety_cap: int = 8) -> str:
    # TODO: implement the agentic loop described above and return the final text
    raise NotImplementedError

In [ ]:
# Exercise 5: check
def _check_ex5():
    calls = []
    impls = {"get_weather": lambda city: calls.append(("w", city)) or {"city": city, "temp_c": 18},
             "get_time": lambda city: calls.append(("t", city)) or {"city": city, "time": "14:05"}}
    tools = [{"name": n, "description": n, "input_schema": {"type": "object"}} for n in impls]

    # (a) text "Done!" + two tool calls in ONE response, then end_turn
    llm = FakeLLM([
        fake_msg("tool_use", say("Done! Let me also check both."), call("get_weather", "t1", city="Oslo"),
                 call("get_time", "t2", city="Oslo")),
        fake_msg("end_turn", say("Oslo: 18°C at 14:05.")),
    ])
    out = run_loop(llm, "Weather and time in Oslo?", tools, impls)
    assert out == "Oslo: 18°C at 14:05.", "return the joined text of the end_turn response (don't stop on 'Done!')"
    assert len(llm.seen) == 2, "expected exactly 2 model calls"
    assert calls == [("w", "Oslo"), ("t", "Oslo")], "run EVERY tool_use block in the response, in order"
    second = llm.seen[1]
    assert second[0] == {"role": "user", "content": "Weather and time in Oslo?"}, "start with the user task"
    assert second[-2]["role"] == "assistant", "append the assistant turn (with its tool_use blocks) to history"
    last = second[-1]
    assert last["role"] == "user" and isinstance(last["content"], list), "send tool results as one user message"
    ids = [b.get("tool_use_id") for b in last["content"] if b.get("type") == "tool_result"]
    assert ids == ["t1", "t2"], "one tool_result per tool_use, all in the SAME user message, ids matching"

    # (b) any other stop_reason must raise, not return partial text
    try:
        run_loop(FakeLLM([fake_msg("max_tokens", say("The answer is"))]), "q", tools, impls)
    except RuntimeError:
        pass
    else:
        raise AssertionError("stop_reason 'max_tokens' must raise RuntimeError")

    # (c) the cap is a safety net: hitting it raises
    looping = FakeLLM([fake_msg("tool_use", call("get_time", f"x{i}", city="Rome")) for i in range(3)])
    try:
        run_loop(looping, "q", tools, impls, safety_cap=3)
    except RuntimeError:
        pass
    else:
        raise AssertionError("reaching safety_cap must raise RuntimeError, not return")
    assert len(looping.seen) == 3, "safety_cap=3 means at most 3 model calls"


check_ex5 = lambda: _record("ex5", _check_ex5)  # noqa: E731
check_ex5()

### Exercise 6 — Trace subagent activity in a message stream

Write `summarize_trace(messages)` for a list of Agent SDK messages. Return:

- `"spawned"`: the `subagent_type` of every subagent-tool call, in order. Match both `"Agent"` (current) and `"Task"` (older), and use `"general-purpose"` when `subagent_type` is missing.
- `"parallel"`: `True` if any single assistant response (same `message_id`) spawned two or more subagents
- `"subagent_messages"`: how many messages have a `parent_tool_use_id`
- `"outcome"`: the `ResultMessage.subtype`, or `None` if the stream has no `ResultMessage`

In [ ]:
# Exercise 6: your code
def summarize_trace(messages: list) -> dict:
    # TODO: return {"spawned": [...], "parallel": bool, "subagent_messages": int, "outcome": str | None}
    raise NotImplementedError

In [ ]:
# Exercise 6: check
def _spawn(msg_id, tool_id, name, subagent_type=None, parent=None):
    inp = {"description": "task", "prompt": "..."}
    if subagent_type:
        inp["subagent_type"] = subagent_type
    return AssistantMessage(model="m", message_id=msg_id, parent_tool_use_id=parent,
                            content=[ToolUseBlock(id=tool_id, name=name, input=inp)])


def _check_ex6():
    run = [
        SystemMessage(subtype="init", data={"tools": ["Task", "Read"]}),
        _spawn("m1", "a1", "Agent", "web-researcher"),
        _spawn("m1", "a2", "Agent", "web-researcher"),
        AssistantMessage(model="m", parent_tool_use_id="a1",
                         content=[ToolUseBlock(id="s1", name="WebSearch", input={"query": "x"})]),
        UserMessage(content=[ToolResultBlock(tool_use_id="s1", content="...")], parent_tool_use_id="a1"),
        _spawn("m2", "a3", "Task"),                                   # legacy name, no subagent_type
        ResultMessage(subtype="error_max_budget_usd", duration_ms=1, duration_api_ms=1, is_error=True,
                      num_turns=5, session_id="s"),
    ]
    out = summarize_trace(run)
    assert isinstance(out, dict), "return a dict"
    assert out.get("spawned") == ["web-researcher", "web-researcher", "general-purpose"], \
        "match BOTH 'Agent' and 'Task'; a missing subagent_type means 'general-purpose'"
    assert out.get("parallel") is True, "two spawns sharing one message_id = one response = parallel"
    assert out.get("subagent_messages") == 2, "count messages whose parent_tool_use_id is set"
    assert out.get("outcome") == "error_max_budget_usd", "outcome is the ResultMessage.subtype"
    seq = summarize_trace([_spawn("m1", "a1", "Agent", "x"), _spawn("m2", "a2", "Agent", "y")])
    assert seq.get("parallel") is False and seq.get("outcome") is None, \
        "spawns in different responses are sequential; no ResultMessage means outcome None"


check_ex6 = lambda: _record("ex6", _check_ex6)  # noqa: E731
check_ex6()

### Exercise 7 — An approval callback (`can_use_tool`)

Write the async callback `approval_gate(tool_name, input_data, context)` that a `ClaudeAgentOptions(can_use_tool=...)` would call. The helper cell defines `READ_ONLY_TOOLS`, `DESTRUCTIVE_PATTERNS`, and a scripted `ask_reviewer(tool_name, tool_input)` that returns `{"approved": bool, "edit": dict | None, "reason": str}`.

- `tool_name` in `READ_ONLY_TOOLS`: return `PermissionResultAllow(updated_input=input_data)` without asking anyone (defense in depth: these are normally pre-approved anyway)
- `Bash` whose `command` matches any `DESTRUCTIVE_PATTERNS` regex (case-insensitive): return `PermissionResultDeny(message=..., interrupt=True)`. Policy denies these, so **don't** ask the reviewer
- anything else: `await ask_reviewer(...)`. If approved, allow with the reviewer's `edit` applied on top of the input. If rejected, deny with a message that includes the reviewer's `reason` (and leave `interrupt` at `False`, so Claude can adapt)
- never mutate `input_data`

Remember the [evaluation order](README.md#37-human-approval-for-irreversible-actions): this callback only sees calls that no hook, rule, or mode resolved, so a tool you want reviewed must not be in `allowed_tools`.

In [ ]:
# Exercise 7: helpers (offline). A scripted reviewer stands in for your review UI.
import re

READ_ONLY_TOOLS = {"Read", "Glob", "Grep", "mcp__crm__lookup_customer"}
DESTRUCTIVE_PATTERNS = [r"\brm\s+-rf\b", r"\bdrop\s+table\b", r"\bgit\s+push\b.*--force"]   # match case-insensitively
REVIEWER_CALLS: list[str] = []
REVIEWER_SCRIPT = {
    "mcp__mail__send": {"approved": True, "edit": {"cc": "support-lead@example.com"}, "reason": ""},
    "mcp__deploy__release": {"approved": False, "edit": None, "reason": "release freeze until Monday"},
    "Bash": {"approved": True, "edit": None, "reason": ""},
}


async def ask_reviewer(tool_name: str, tool_input: dict) -> dict:
    """Returns {"approved": bool, "edit": dict | None, "reason": str}."""
    REVIEWER_CALLS.append(tool_name)
    return REVIEWER_SCRIPT.get(tool_name, {"approved": False, "edit": None, "reason": "no reviewer for this tool"})

In [ ]:
# Exercise 7: your code
async def approval_gate(tool_name: str, input_data: dict, context: ToolPermissionContext):
    # TODO: read-only -> allow; destructive Bash -> deny with interrupt=True (no reviewer);
    #       everything else -> await ask_reviewer(...) and allow (with edits) or deny with the reason
    raise NotImplementedError

In [ ]:
# Exercise 7: check (async, so it is awaited)
async def _check_ex7():
    ctx = ToolPermissionContext(tool_use_id="toolu_7")

    async def call(tool_name, tool_input):
        before = copy.deepcopy(tool_input)
        REVIEWER_CALLS.clear()
        result = await approval_gate(tool_name, tool_input, ctx)
        assert tool_input == before, "never mutate input_data; build a new dict for updated_input"
        assert isinstance(result, (PermissionResultAllow, PermissionResultDeny)), \
            "return PermissionResultAllow or PermissionResultDeny"
        return result, list(REVIEWER_CALLS)

    r, seen = await call("Read", {"file_path": "notes.md"})
    assert isinstance(r, PermissionResultAllow) and not seen, "read-only tools are allowed without asking the reviewer"
    assert r.updated_input == {"file_path": "notes.md"}, "allow read-only calls with the original input"

    for cmd in ["rm -rf /var/data", "psql -c 'DROP TABLE users'", "git push origin main --force"]:
        r, seen = await call("Bash", {"command": cmd})
        assert isinstance(r, PermissionResultDeny), f"{cmd!r} is destructive: deny it"
        assert r.interrupt is True, "a destructive command should also interrupt the run (interrupt=True)"
        assert not seen, "destructive commands are denied by policy; don't ask the reviewer to click them through"
        assert r.message, "give Claude a message explaining the denial"

    r, seen = await call("Bash", {"command": "ls -la"})
    assert seen == ["Bash"] and isinstance(r, PermissionResultAllow), "other Bash commands go to the reviewer"

    email = {"to": "ana@example.com", "body": "Your refund is on its way."}
    r, seen = await call("mcp__mail__send", email)
    assert seen == ["mcp__mail__send"], "side-effecting tools must ask the reviewer"
    assert isinstance(r, PermissionResultAllow), "the reviewer approved: allow"
    assert r.updated_input == {**email, "cc": "support-lead@example.com"}, \
        "apply the reviewer's edit on top of the original input"

    r, seen = await call("mcp__deploy__release", {"version": "2.4.0"})
    assert isinstance(r, PermissionResultDeny) and seen == ["mcp__deploy__release"], "the reviewer rejected: deny"
    assert "release freeze" in r.message, "pass the reviewer's reason to Claude in the deny message"
    assert r.interrupt is False, "a reviewer rejection should let Claude adapt, not interrupt the run"


async def check_ex7():
    return await _record_async("ex7", _check_ex7)


await check_ex7()

### Optional live stretch (live, ungraded, about $0.05–$0.15)

Run your Exercise 4 hook inside a real agent and ask it for a $750 refund. The hook should deny the call, Claude should adapt (usually by escalating), and `REFUNDS` should stay empty. This cell needs `ANTHROPIC_API_KEY`, and it calls `require_live()` so the Agent SDK never starts without one.

In [ ]:
# (live, optional, ungraded) Your Exercise 4 hook inside a real agent run. Capped at $0.15
require_live()
import tempfile

REFUNDS: list[dict] = []


@tool("process_refund", "Refund an order. Inputs: order_id, amount (USD).", {"order_id": str, "amount": float})
async def process_refund(args):
    REFUNDS.append(args)
    return {"content": [{"type": "text", "text": json.dumps({"refund_id": "RF-1", **args})}]}


@tool("escalate_to_human", "Hand the case to a human. Inputs: customer_id, amount, reason.",
      {"customer_id": str, "amount": float, "reason": str})
async def escalate_to_human(args):
    return {"content": [{"type": "text", "text": json.dumps({"ticket": "T-1", "status": "queued"})}]}


opts = ClaudeAgentOptions(
    model=MODEL, cwd=tempfile.mkdtemp(), setting_sources=[], permission_mode="dontAsk",
    system_prompt="You are a support agent for customer C-7.",
    mcp_servers={"support": create_sdk_mcp_server(name="support", tools=[process_refund, escalate_to_human])},
    allowed_tools=["mcp__support__*", "ToolSearch"],
    disallowed_tools=["Bash", "Write", "Edit", "WebSearch", "WebFetch"],
    hooks={"PreToolUse": [HookMatcher(matcher="mcp__support__process_refund", hooks=[enforce_refund_policy])]},
    max_turns=6, max_budget_usd=0.15,
)
try:
    async for m in query(prompt="Refund order A-1002 for $750, it arrived broken.", options=opts):
        if isinstance(m, ResultMessage):
            print(m.subtype, m.result)
except Exception as err:
    print("Run ended with an error:", err)
print("refunds executed:", REFUNDS, "(should be empty)")

## Part C — Architecture scenario

**Scenario: insurance claims intake.** An insurer wants an agent system that takes a new property-damage claim (a free-text description, photos, and a policy number), checks coverage against the policy, flags likely fraud, and either pays the claim or routes it to an adjuster.

Constraints:

- Payouts above **$5,000** require a human adjuster. A single wrong payout is a financial and regulatory incident.
- The claimant's identity must be verified before any policy data is read.
- Two third-party data vendors (weather history, contractor price index) return dates as epoch seconds, ISO strings, or `"Mar 5, 2026"`.
- Peak load is about 2,000 claims a day. The business wants low cost per claim and clear audit trails.
- Complex claims can take several days, with new documents arriving in between.

Write your design (around 250–400 words), covering:

1. One agent or coordinator + subagents? Name each agent, its `description` in one line, its `tools`, and its `model`.
2. Which rules go in **hooks**, and which in **prompts**, with the reason for each.
3. How context reaches each subagent, and what runs in parallel.
4. Error handling between subagents and the coordinator.
5. Cost and safety limits.
6. Session strategy across multiple days.

*Your answer:*

…

<details><summary><b>Rubric: what a strong answer contains (open after writing yours)</b></summary>

- **Topology:** a coordinator plus 3–4 focused subagents, for example `identity-verifier` (`get_claimant`, `verify_identity`), `coverage-analyst` (`get_policy`, `get_claim_docs`), `fraud-screener` (`vendor_weather`, `vendor_price_index`, `claims_history`), `payout-agent` (`issue_payout`, `route_to_adjuster`). Each has explicit `tools` (none omitted), a cheaper model such as Haiku for narrow lookups, and a stronger model for the coordinator or coverage reasoning. The answer explains that a single agent with every tool would choose tools less reliably.
- **Hooks for hard rules:** a `PreToolUse` on `issue_payout` denies amounts above $5,000 (checked against the system's own claim amount, not the model's argument) and redirects to `route_to_adjuster`. A `PreToolUse` precondition denies policy and claim reads until identity verification has succeeded, tracked through a `PostToolUse` on `verify_identity`. A `PostToolUse` normalizes vendor dates centrally. **Prompts** handle tone, the claimant summary style, and soft triage heuristics. The answer names the cost-of-failure test.
- **Context passing:** each `Agent` call carries everything the subagent needs (claim ID, structured claim facts, relevant documents or IDs, prior findings verbatim, output JSON schema). Coverage analysis and fraud screening run **in parallel** after identity verification. Payout runs sequentially, after both.
- **Errors:** subagents retry transient vendor timeouts locally, then return structured failures (type, attempted call, partial results, alternatives). "No prior claims" is a valid finding, not an error. The coordinator decides whether to retry, reroute to an adjuster, or continue with partial data, and never treats an empty result as success.
- **Limits:** `max_budget_usd` per claim, `max_turns` as a guardrail, `CLAUDE_CODE_MAX_SUBAGENT_SPAWN_DEPTH=1`, a concurrency cap, `permission_mode="dontAsk"` with explicit `allowed_tools`, `disallowed_tools` for Bash and Write, and `ResultMessage.subtype` handling (for example, `error_max_budget_usd` routes the claim to a human).
- **Approvals:** payouts at or below $5,000 that the policy allows can still go to a reviewer through `can_use_tool` (with `issue_payout` kept **out** of `allowed_tools` and `permission_mode` set explicitly). The $5,000 limit itself stays in the `PreToolUse` hook, so no reviewer can click it through. Long reviews use a hook `defer` and a resumed session rather than a callback that waits for days.
- **Sessions:** store `session_id` per claim. When new documents arrive, resume and **state what changed**. If most earlier findings are stale (the policy was amended), start a new session seeded with a structured claim summary. Use `fork_session` only to compare alternative settlements.
- **Audit:** `SubagentStart`/`SubagentStop` and `PreToolUse` decisions are logged in code, outside the model's context.

</details>

## Scorecard

Run this after Part A and Part B. Re-run any check cell after fixing an exercise, then run the scorecard again.

In [ ]:
# Scorecard
def scorecard(pass_mark: float = 72.0) -> None:
    quiz = RESULTS.get("quiz_pct", 0.0)
    exercises = [f"ex{i}" for i in range(1, 8)]
    passed = sum(bool(RESULTS.get(e)) for e in exercises)
    ex_pct = 100 * passed / len(exercises)
    overall = (quiz + ex_pct) / 2
    print(f"Quiz:      {quiz:5.1f}%")
    print(f"Exercises: {passed}/{len(exercises)} = {ex_pct:5.1f}%   " +
          " ".join(f"{e}:{'✓' if RESULTS.get(e) else '✗'}" for e in exercises))
    print(f"Overall:   {overall:5.1f}%  (pass mark {pass_mark:.0f}%)")
    print("PASS: on to Chapter 4." if overall >= pass_mark else "RETRY: reread the README sections you missed, then try again.")


scorecard()

## Solutions (spoilers)

Open these only after you have attempted everything.

<details><summary><b>Part A: quiz answers with explanations</b></summary>

- **q1: C.** Completion in the Agent SDK is `ResultMessage.subtype`. Iterate to completion, because trailing events can follow the result. A only moves the cap, B is still text matching, and D repeats the anti-pattern "response has text = done".
- **q2: A.** `allowed_tools` is an allow list, not a sandbox. In `bypassPermissions`, every unlisted tool is approved. Fix it with `disallowed_tools=["Bash"]`, a hook, or `dontAsk` with explicit `allowed_tools`.
- **q3: D.** An omitted `tools` field inherits every tool available to subagents. Prompts (A) are probabilistic, the coordinator's allow list (B) isn't the subagent's tool set, and the model choice (C) is unrelated.
- **q4: B.** The dict key is the name, and the fields are `description`, `prompt`, and `tools`. Multi-word fields keep camelCase (`maxTurns`), so C raises `TypeError`. A is the guide's pseudocode, and D isn't an option.
- **q5: C.** Subagents start fresh. The `Agent` prompt string is the only channel from the parent. B can't work because there is nothing to remember, A and D treat symptoms, and D also wastes turns.
- **q6: A.** Matchers see only the tool name. Here the parentheses even switch the matcher to the regex path, so it would never match. Inspect `tool_input` in the callback. PostToolUse (C) fires after the money has already moved.
- **q7: D.** When hooks disagree, the most restrictive decision wins (`deny` > `defer` > `ask` > `allow`). Hooks run before the permission mode, so a hook deny holds even in `bypassPermissions`.
- **q8: B.** Partition before you delegate. That fixes the cause and saves the tokens. Deduplicating afterwards (A) still pays for the duplicate work. C bypasses the hub, and D removes context isolation and parallelism.
- **q9: C.** "0 results" is a valid finding, while a timeout needs a retry or reroute decision. A disguises a failure as success, B throws away context, and D overreacts to a partial failure.
- **q10: B.** Most earlier observations are stale, so a fresh session seeded with a structured summary and a list of changes is more reliable. A reasons over outdated tool results. C and D still carry the stale context.
- **q11: D.** If you don't set a mode, Claude Code uses a `defaultMode` from the loaded settings, or else its built-in default, which can be `auto`. Auto mode lets a classifier decide and drops broad allow rules. Always pass the mode you designed for. A, B and C describe behavior that doesn't exist.
- **q12: A.** Hooks run first, but a hook `allow` doesn't skip the deny and ask rules that follow. A scoped deny rule blocks matching calls in every mode. Only a hook `deny` ends the evaluation early.

</details>

<details><summary><b>Exercise 1: build_research_agents</b></summary>

The key is the name. `tools` is explicit for both, and `maxTurns` is camelCase.

```python
def build_research_agents() -> dict[str, AgentDefinition]:
    return {
        "web-researcher": AgentDefinition(
            description=("Finds and summarizes external web sources for ONE assigned subtopic. "
                         "Use for fact-finding on the open web; never for writing the final report."),
            prompt=("Research only the subtopic in your prompt. Return a JSON list of "
                    "{claim, source_url, quote, published_date}. Report gaps explicitly."),
            tools=["WebSearch", "WebFetch"],
            model="haiku",
            maxTurns=8,
        ),
        "synthesizer": AgentDefinition(
            description=("Merges research findings passed in its prompt into a cited report. "
                         "Use after all research is done; it does no searching of its own."),
            prompt=("Write the report using ONLY the findings provided. Keep each claim's source. "
                    "Return JSON {sections: [...], conflicts: [...], gaps: [...]}."),
            tools=["Read"],
            model="inherit",
        ),
    }
```

</details>

<details><summary><b>Exercise 2: build_coordinator_options</b></summary>

`allowed_tools` pre-approves, `disallowed_tools` removes, `dontAsk` denies everything else, and `env` values are strings.

```python
def build_coordinator_options(agents: dict[str, AgentDefinition]) -> ClaudeAgentOptions:
    return ClaudeAgentOptions(
        model=MODEL,
        system_prompt=("You coordinate research. Partition the topic into non-overlapping subtopics that cover "
                       "all major areas, delegate them in parallel, pass findings verbatim to the synthesizer, "
                       "and report gaps."),
        agents=agents,
        allowed_tools=["Agent", "WebSearch", "WebFetch", "Read"],
        disallowed_tools=["Bash"],
        permission_mode="dontAsk",
        max_turns=25,
        max_budget_usd=1.50,
        env={"CLAUDE_CODE_MAX_SUBAGENT_SPAWN_DEPTH": "1"},
    )
```

</details>

<details><summary><b>Exercise 3: assign_tools</b></summary>

The coordinator only gets `Agent` plus escalation. Nobody gets `WebSearch`, `WebFetch`, or `Bash`, and the writer gets one narrow tool.

```python
def assign_tools() -> dict[str, set[str]]:
    return {
        "coordinator": {"Agent", "mcp__crm__escalate_to_human"},
        "order-specialist": {"mcp__crm__get_customer", "mcp__crm__lookup_order"},
        "refund-specialist": {"mcp__crm__lookup_order", "mcp__crm__process_refund", "mcp__crm__escalate_to_human"},
        "reply-writer": {"mcp__kb__verify_fact"},
    }
```

</details>

<details><summary><b>Exercise 4: enforce_refund_policy</b></summary>

Parse defensively and fail closed. The decision lives inside `hookSpecificOutput`, and the reason tells Claude what to do next.

```python
REFUND_LIMIT_USD = 500


def _deny(reason: str) -> dict:
    return {"hookSpecificOutput": {"hookEventName": "PreToolUse",
                                   "permissionDecision": "deny", "permissionDecisionReason": reason}}


async def enforce_refund_policy(input_data: dict, tool_use_id: str | None, context: Any) -> dict:
    try:
        amount = float(input_data["tool_input"].get("amount"))
    except (TypeError, ValueError):
        return _deny("Refund amount is missing or not a number. Look up the order total first.")
    if amount > REFUND_LIMIT_USD:
        return _deny(f"Refunds over ${REFUND_LIMIT_USD} need human approval. "
                     "Call mcp__support__escalate_to_human with customer_id, amount and reason.")
    return {}
```

</details>

<details><summary><b>Exercise 5: run_loop</b></summary>

`stop_reason` drives the loop. All tool results go back in one user message, and the cap raises an error instead of returning.

```python
def run_loop(llm, task: str, tools: list[dict], impls: dict, safety_cap: int = 8) -> str:
    messages = [{"role": "user", "content": task}]
    for _ in range(safety_cap):
        resp = llm.messages.create(model=MODEL, max_tokens=512, tools=tools, messages=messages)
        messages.append({"role": "assistant", "content": resp.content})
        if resp.stop_reason == "end_turn":
            return "".join(b.text for b in resp.content if b.type == "text")
        if resp.stop_reason == "tool_use":
            messages.append({"role": "user", "content": [
                {"type": "tool_result", "tool_use_id": b.id, "content": json.dumps(impls[b.name](**b.input))}
                for b in resp.content if b.type == "tool_use"]})
            continue
        raise RuntimeError(f"Unhandled stop_reason: {resp.stop_reason}")
    raise RuntimeError("Safety cap reached")
```

</details>

<details><summary><b>Exercise 6: summarize_trace</b></summary>

Match both tool names. `message_id` groups the blocks of one response, and `parent_tool_use_id` marks messages from inside subagents.

```python
def summarize_trace(messages: list) -> dict:
    spawned, per_response, inside, outcome = [], {}, 0, None
    for m in messages:
        if isinstance(m, AssistantMessage):
            for b in m.content:
                if isinstance(b, ToolUseBlock) and b.name in ("Agent", "Task"):
                    spawned.append(b.input.get("subagent_type") or "general-purpose")
                    if m.message_id:
                        per_response[m.message_id] = per_response.get(m.message_id, 0) + 1
        if getattr(m, "parent_tool_use_id", None):
            inside += 1
        if isinstance(m, ResultMessage):
            outcome = m.subtype
    return {"spawned": spawned, "parallel": any(n >= 2 for n in per_response.values()),
            "subagent_messages": inside, "outcome": outcome}
```

</details>

<details><summary><b>Exercise 7: approval_gate</b></summary>

Policy first (read-only allow, destructive deny with `interrupt=True`), then the human. Edits go into a new dict, and the deny message carries the reason so Claude can adapt.

```python
async def approval_gate(tool_name: str, input_data: dict, context: ToolPermissionContext):
    if tool_name in READ_ONLY_TOOLS:
        return PermissionResultAllow(updated_input=input_data)
    if tool_name == "Bash":
        command = str(input_data.get("command", ""))
        if any(re.search(p, command, re.IGNORECASE) for p in DESTRUCTIVE_PATTERNS):
            return PermissionResultDeny(message=f"Destructive command blocked by policy: {command!r}.",
                                        interrupt=True)
    review = await ask_reviewer(tool_name, input_data)
    if review["approved"]:
        return PermissionResultAllow(updated_input={**input_data, **(review.get("edit") or {})})
    return PermissionResultDeny(message=f"A reviewer rejected {tool_name}: {review['reason']}. "
                                        "Tell the user and propose an alternative.")
```

</details>

Next chapter: [Chapter 4 — Model Context Protocol](../04-model-context-protocol/README.md)